# Train Open Duck Mini v2 walking policies for the MaixCAM2

This notebook trains any or all of the policy **flavors** in `training/odm_train/flavors.py`, evaluates them in MuJoCo, and packages each one as `ONNX + JSON profile` for the robot.

It runs on **Google Colab** (Runtime → Change runtime type → GPU) or **locally in Jupyter** on any NVIDIA GPU (for example an RTX 3050).

| Flavor | What it is | Runs on the robot? |
|---|---|---|
| `follow_flat_backlash` | **Recommended.** Follow-mode commands, the head moves, leg backlash, strong randomization | yes |
| `follow_rough_backlash` | Same on rough ground. Learns slower, sturdier on carpet and tiles | yes |
| `follow_flat` | Follow mode without backlash joints | yes |
| `stock_flat`, `stock_flat_backlash`, `stock_rough_backlash` | Upstream Joystick task retrained as-is, as a baseline | yes |
| `standing_flat` | Upstream standing/balance task (85-value observation) | not yet, experimental |

**GPU tips**
- Colab free T4: works, but a full 300M-step run is longer than a free session. Checkpoints go to Google Drive and the next run resumes automatically, so just re-run the notebook after a disconnect.
- Colab Pro L4/A100: a few hours per flavor.
- RTX 3050 8 GB: try `NUM_ENVS = 4096` first. 4 GB laptop cards: `NUM_ENVS = 2048`.
- Start with `NUM_TIMESTEPS = 50_000_000` to check a flavor learns before you commit to a full run.

In [ ]:
#@title Settings
FLAVORS_TO_TRAIN = ["follow_flat_backlash"]  # or "all", or e.g. ["follow_flat_backlash", "follow_rough_backlash", "stock_flat"]
NUM_TIMESTEPS = 300_000_000   # per flavor, total (resumed steps count)
NUM_ENVS = None               # None = 8192. Lower it if you run out of GPU memory (4096, 2048)
NUM_EVALS = 30                # checkpoints per run; more = less work lost on a disconnect
REPO_BRANCH = "main"          # odm-maixcam2 branch to train from
EVAL_EPISODES = 40            # sim_eval episodes per policy (0 = skip evaluation)

In [ ]:
#@title Where to work (Drive on Colab, ./duck_runs locally)
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = "/content/drive/MyDrive/odm_maixcam2_runs"
    WORK = "/content"
else:
    OUT_ROOT = os.path.abspath("duck_runs")
    WORK = os.path.abspath("duck_work")
os.makedirs(OUT_ROOT, exist_ok=True)
os.makedirs(WORK, exist_ok=True)
print("checkpoints ->", OUT_ROOT)
try:
    subprocess.run(["nvidia-smi"], check=False)
except FileNotFoundError:
    print("nvidia-smi not found: is an NVIDIA driver installed?")

In [ ]:
#@title Get the code
# Open_Duck_Playground pinned to the commit this repo's trainer was tested against.
ODP_COMMIT = "b9be205ac64488c23504ca42e5ec790337adeec3"
def sh(cmd, cwd=None):
    print("$", cmd)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)

ODP = os.path.join(WORK, "Open_Duck_Playground")
REPO = os.path.join(WORK, "odm-maixcam2")
STOCK = os.path.join(WORK, "Open_Duck_Mini")
if not os.path.exists(ODP):
    sh(f"git clone -q https://github.com/apirrone/Open_Duck_Playground {ODP}")
sh(f"git fetch -q origin && git checkout -q {ODP_COMMIT}", cwd=ODP)
if not os.path.exists(REPO):
    sh(f"git clone -q https://github.com/aizatrosli/odm-maixcam2 {REPO}")
sh(f"git fetch -q origin {REPO_BRANCH} && git checkout -q FETCH_HEAD", cwd=REPO)
if not os.path.exists(os.path.join(STOCK, "BEST_WALK_ONNX_2.onnx")):  # stock policy, for comparison
    sh(f"git clone -q --depth 1 --filter=blob:none --sparse https://github.com/apirrone/Open_Duck_Mini {STOCK}")
    sh("git sparse-checkout set --no-cone /BEST_WALK_ONNX_2.onnx", cwd=STOCK)

In [ ]:
#@title Install pinned dependencies (a few minutes)
# Newer mujoco_playground releases removed modules the env imports, so these versions are pinned.
# Tested together: training, checkpoint/resume and ONNX export.
PINS = [
    "jax[cuda12]==0.8.0", "mujoco==3.3.7", "mujoco-mjx==3.3.7", "brax==0.13.0", "playground==0.0.5",
    "flax==0.12.0", "orbax-checkpoint==0.12.8", "ml-collections==1.1.0",
    "tensorflow==2.21.0", "tf2onnx==1.17.0", "onnx==1.23.2", "onnxruntime==1.31.1",
    "tensorboardX==2.6.5", "jaxtyping", "mediapy",
]
sh(f"{sys.executable} -m pip install -q " + " ".join(f"'{p}'" for p in PINS))

In [ ]:
#@title Check JAX sees the GPU
out = subprocess.run([sys.executable, "-c", "import jax; print(jax.devices())"], capture_output=True, text=True)
print(out.stdout, out.stderr[-2000:])
assert "cuda" in out.stdout.lower() or "gpu" in out.stdout.lower(), "No GPU found: training on CPU would take months."

In [ ]:
#@title Train (re-run after a disconnect: each flavor resumes from its newest checkpoint)
ENV = dict(os.environ, PYTHONPATH=f"{ODP}:{os.path.join(REPO, 'training')}", XLA_PYTHON_CLIENT_MEM_FRACTION="0.9")
sys.path.insert(0, os.path.join(REPO, "training"))
from odm_train.flavors import FLAVORS, DEPLOYABLE
flavors = list(FLAVORS) if FLAVORS_TO_TRAIN == "all" else list(FLAVORS_TO_TRAIN)
for f in flavors:
    assert f in FLAVORS, f"unknown flavor {f}; choose from {list(FLAVORS)}"

def run_streaming(cmd):
    p = subprocess.Popen(cmd, cwd=ODP, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        if "STEP:" in line or "Flavor" in line or "Resuming" in line or "Saving" in line or "Error" in line or "nothing to do" in line:
            print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"training failed ({p.returncode}); rerun the cell without filters to see the log")

for f in flavors:
    print(f"\n===== {f} =====")
    cmd = [sys.executable, "-m", "odm_train.train", "--flavor", f, "--resume",
           "--output_dir", os.path.join(OUT_ROOT, f), "--num_timesteps", str(NUM_TIMESTEPS),
           "--num_evals", str(NUM_EVALS), "--jax_cache", os.path.join(OUT_ROOT, "jax_cache")]
    if NUM_ENVS:
        cmd += ["--num_envs", str(NUM_ENVS)]
    run_streaming(cmd)

In [ ]:
#@title Training curves
%load_ext tensorboard
%tensorboard --logdir "$OUT_ROOT"

In [ ]:
#@title Evaluate in MuJoCo against the stock policy (follow-style commands, head moving)
import glob, json
def latest_onnx(flavor):
    try:
        with open(os.path.join(OUT_ROOT, flavor, "progress.json")) as fh:
            return os.path.join(OUT_ROOT, flavor, json.load(fh)["latest_onnx"])
    except (OSError, KeyError, ValueError):
        return None

to_eval = [os.path.join(STOCK, "BEST_WALK_ONNX_2.onnx")]
to_eval += [p for p in (latest_onnx(f) for f in flavors if f in DEPLOYABLE) if p]
if EVAL_EPISODES:
    for args in ([], ["--no_head"]):
        print("head moving" if not args else "head still")
        out = subprocess.run([sys.executable, "-m", "odm_train.sim_eval", *to_eval, "-n", str(EVAL_EPISODES), *args],
                             cwd=ODP, env=ENV, capture_output=True, text=True)
        print("\n".join(l for l in out.stdout.splitlines() if l.startswith("{")) or out.stderr[-3000:])
print("\nKeep a flavor only if it falls less and has lower wz_rms than the stock policy with the head moving.")

In [ ]:
#@title Package each flavor for the robot (ONNX + JSON profile)
import shutil, zipfile
export_dir = os.path.join(OUT_ROOT, "export")
os.makedirs(export_dir, exist_ok=True)
zips = []
for f in flavors:
    onnx_path = latest_onnx(f)
    if not onnx_path:
        print(f"{f}: no checkpoint yet"); continue
    z = os.path.join(export_dir, f"{f}.zip")
    with zipfile.ZipFile(z, "w") as zf:
        zf.write(onnx_path, f"{f}.onnx")
        zf.write(onnx_path[:-5] + ".json", f"{f}.json")
    zips.append(z)
    print(f"{f}: {os.path.basename(onnx_path)} -> {z}")
if IN_COLAB:
    from google.colab import files
    for z in zips:
        files.download(z)

## On the robot

Unzip a flavor next to the runtime (keep the `.onnx` and `.json` together, same name) and run it on the CPU first:

```bash
python3 scripts/walk.py --policy ~/follow_flat_backlash.onnx --follow --save_obs obs.npy
```

**NPU conversion** needs Docker and the Pulsar2 image, which Colab does not have, so it runs on a PC:

```bash
tools/convert/convert_policy.sh follow_flat_backlash.onnx obs.npy
```

Use `obs.npy` recorded with the new policy, because its observations differ slightly from the stock policy's. `standing_flat` uses an 85-value observation, which `scripts/walk.py` does not support yet.